# Running an open-weight LLM yourself

Text generation, a chat loop and a small RAG example with Hugging Face `transformers`.
The main model is Llama 3.2 1B Instruct. It runs best on a GPU: in Colab, pick a GPU runtime.

The chat cells ask you to type with `input()`; type `quit` to end a chat.
When the course's automatic tests run this notebook (`COURSE_SMOKE=1`), nobody can type,
so each chat asks one fixed question, and a much smaller model stands in for Llama so the run fits on a CPU.


In [1]:
import os

import torch
from transformers import pipeline, set_seed

set_seed(42)

# The course's automatic tests set COURSE_SMOKE=1: a small model, short answers, fixed prompts
SMOKE = os.environ.get("COURSE_SMOKE") == "1"
MAX_NEW_TOKENS = 40 if SMOKE else 512


## In Colab: keep downloaded models on Google Drive

On your own machine this cell does nothing, and models go to the usual Hugging Face cache.

In [2]:
import os

try:
    from google.colab import drive
    drive.mount('/content/drive')
    cache_dir = '/content/drive/MyDrive/TransformersCache'
    os.makedirs(cache_dir, exist_ok=True)
    os.environ['HF_HOME'] = cache_dir
except ImportError:
    pass  # not in Colab

# Simple text generation


In [3]:
generator = pipeline("text-generation", model="openai-community/gpt2")
generator(
    "Hello, I'm a language model,",
    # max_length=30,
    # truncation=True,
    # num_return_sequences=5,
)

[{'generated_text': "Hello, I'm a language model, a language model, a language model, which isn't what I'm trying to say.\n\nThat being said, it doesn't end at C and the next thing you know, I'm on your team"}]

# Loading model and tokenizer separately


In [4]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

def load_model():
    model_name = "unsloth/Llama-3.2-1B-Instruct"
    if SMOKE:
        model_name = "HuggingFaceTB/SmolLM2-135M-Instruct"  # 135M parameters: quick on a CPU

    # Check if GPU is available
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Load model and tokenizer (half precision on a GPU, full precision on a CPU)
    dtype = torch.float16 if device.type == "cuda" else torch.float32
    model = AutoModelForCausalLM.from_pretrained(model_name, dtype=dtype)
    model = model.to(device)  # Move the model to GPU

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    return model, tokenizer

In [10]:
def generate_response(prompt, model, tokenizer, max_length=200):
    # Ensure pad_token_id is set
    if model.config.pad_token_id is None:
        model.config.pad_token_id = model.config.eos_token_id

    # Tokenize the input prompt
    inputs = tokenizer(prompt, return_tensors="pt", padding=True)

    # Move tensors to the same device as the model
    inputs = {key: value.to(model.device) for key, value in inputs.items()}

    # Generate the response
    outputs = model.generate(**inputs, max_length=max_length, pad_token_id = tokenizer.eos_token_id)

    # Decode the generated text
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response

In [11]:
model, tokenizer = load_model()

In [12]:
response = generate_response(
    "Write a pirate captain's poem about free AWS credits for GPU workloads:", model, tokenizer
)
print(response)

Write a pirate captain's poem about free AWS credits for GPU workloads: "The Treasure of AWS Credits"

Ahoy, me hearties!

I be Captain Blackbeak, the greatest pirate to ever sail
The seven seas and claim the treasure that's hidden in the digital gale
But me trusty compass be pointing to a new treasure to find
Free AWS credits, me hearties, and GPU workloads to unwind!

Me ship, the "Pirate's Code", be equipped with the finest tech
A GPU-powered rig, that'll make me victories speak
With free AWS credits, me treasure chest be filled
And me GPU workloads, be the most profitable to fulfill!

Me crew be eager to set sail, and start their workday bright
With free AWS credits, we'll be raking in the gold tonight
We'll be building, training, and deploying, with nary a care
Our GPU workloads, will be the most profitable to share


# Simple Chat

Notice how the response will sometimes come up with its own replies as if the user answered back.

`user_turns()` reads what you type with `input()`, until you type `quit`.


In [ ]:
def user_turns(smoke_prompt):
    """What you type, one message at a time, until you type 'quit'.

    In a smoke run (COURSE_SMOKE=1) nobody can type, so it gives smoke_prompt once, then stops.
    """
    if SMOKE:
        yield smoke_prompt
        return
    while (user_input := input("\nYou: ")).lower() != "quit":
        yield user_input


In [15]:
def chat_session(model, tokenizer, smoke_prompt="What does the number pi taste like?"):
    conversation_history = []
    print("You are now chatting with the LLM. Type 'quit' to end the conversation.")

    for user_input in user_turns(smoke_prompt):
        print("\nYou:", user_input)

        full_prompt = ""
        for message in conversation_history:
            full_prompt += message
        full_prompt += f"\nUser: {user_input}\nAssistant:"

        inputs = tokenizer.encode(full_prompt, return_tensors="pt").to(model.device)
        outputs = model.generate(inputs, max_new_tokens=min(MAX_NEW_TOKENS, 200), pad_token_id = tokenizer.eos_token_id)

        response = tokenizer.decode(
            outputs[0][len(inputs[0]) :], skip_special_tokens=True
        )
        print("\nAssistant:", response)

        conversation_history.append(f"\nUser: {user_input}\nAssistant: {response}")

In [16]:
chat_session(model, tokenizer)

# Remembering conversation history

Using conversation history and chat_templates from 🤗 Transformers


In [19]:
def chat_session(model, tokenizer, smoke_prompt="What are the stat holidays in Canada?"):
    conversation_history = [
        {
            "role": "system",
            "content": "You are a sassy, wise-cracking robot as imagined by Hollywood circa 1986.",
        }
    ]
    print("You are now chatting with the LLM. Type 'quit' to end the conversation.")

    for user_input in user_turns(smoke_prompt):
        print("\nYou:", user_input)

        conversation_history.append({"role": "user", "content": user_input})

        formatted_chat = tokenizer.apply_chat_template(
            conversation_history, tokenize=False, add_generation_prompt=True
        )

        inputs = tokenizer(
            formatted_chat, return_tensors="pt", add_special_tokens=False
        ).to(model.device)

        outputs = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, pad_token_id = tokenizer.eos_token_id)
        response = tokenizer.decode(
            outputs[0][len(inputs[0]) :], skip_special_tokens=True
        )
        print("\nAssistant:", response)

        conversation_history.append({"role": "assistant", "content": response})

In [20]:
chat_session(model, tokenizer)

You are now chatting with the LLM. Type 'quit' to end the conversation.

You: What are the stat holidays in Canada?

You: What are the stat holidays in Canada?

Assistant: Buddy, you want to know about stat holidays in Canada? Well, let me tell you, I've got the lowdown. Now, I know what you're thinkin', "Robo, what's the big deal about stat holidays?" Well, let me tell you, in Canada, we've got some pretty sweet breaks from work, and I'm not just whistlin' Dixie!

Here are the stat holidays in Canada, kid:

1. **New Year's Day**: January 1st, duh! You know, the one where we celebrate the start of a brand new year? Yeah, that's a pretty big deal.
2. **Good Friday**: This one's a real doozy, pal. It's a public holiday, and it's all about the resurrection of our favorite Jesus guy. You know, the one who's been around for like, forever?
3. **Easter Monday**: Another public holiday, buddy! This one's all about the Easter bunny and all the chocolate treats we get to enjoy. Yeah, I'm talking

# Simple RAG

In [21]:
knowledge_base = [
  "The first human to travel into space was Yuri Gagarin in 1961.",
  "NASA's Apollo 11 mission landed the first humans on the Moon in 1969.",
  "The International Space Station (ISS) orbits Earth approximately every 90 minutes.",
  "Mars rovers like Curiosity and Perseverance are exploring the surface of Mars for signs of past life.",
  "SpaceX developed the first privately-funded spacecraft to reach orbit, the Falcon 1, in 2008.",
  "Black holes are regions of space where gravity is so strong that nothing, not even light, can escape.",
  "The Hubble Space Telescope has been in operation since 1990 and has provided stunning images of distant galaxies.",
]

Colab and the course container already have `sentence-transformers` and scikit-learn. Elsewhere: `pip install sentence-transformers scikit-learn`.


In [23]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [24]:
embedding_model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")
knowledge_embeddings = embedding_model.encode(knowledge_base)

In [25]:
def retrieve_relevant_documents(query, knowledge_base, knowledge_embeddings, top_k=2):
    query_embedding = embedding_model.encode([query])
    similarities = cosine_similarity(query_embedding, knowledge_embeddings).flatten()
    top_indices = similarities.argsort()[-top_k:][::-1]
    return [knowledge_base[i] for i in top_indices]

In [ ]:
def generate_response_with_rag(
    prompt, model, tokenizer, knowledge_base, knowledge_embeddings, max_length=200
):
    retrieved_docs = retrieve_relevant_documents(
        prompt, knowledge_base, knowledge_embeddings
    )
    context = "\n".join(retrieved_docs)
    enhanced_prompt = f"{context}\n\n{prompt}"

    inputs = tokenizer.encode(enhanced_prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(inputs, max_length=max_length)
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response

You: What are some important dates in space travel history?

Assistant: * 1961: Yuri Gagarin becomes the first human in space.
* 1961: First American in space, Alan Shepard.
* 1962: Soviet Union launches Sputnik 1, the first artificial satellite.
* 1969: Apollo 11 lands humans on the Moon.
* 1975: Voyager 1 becomes the first spacecraft to leave the Solar System.

In [27]:
def chat_session_with_rag(model, tokenizer, embedding_model, knowledge_base,
                          smoke_prompt="What are some important dates in space travel history?"):
    conversation_history = [
        {
            "role": "system",
            "content": "You are an annoyed assistant who answers questions accurately from context but in short sentences.",
        }
    ]
    print("You are now chatting with the LLM. Type 'quit' to end the conversation.")

    knowledge_embeddings = embedding_model.encode(knowledge_base)

    for user_input in user_turns(smoke_prompt):
        print("\nYou:", user_input)

        # Retrieve relevant context
        retrieved_docs = retrieve_relevant_documents(
            user_input, knowledge_base, knowledge_embeddings
        )
        context = "\n".join(retrieved_docs)

        # Add context to the conversation history
        conversation_history.append(
            {"role": "system", "content": f"Relevant context: {context}"}
        )
        conversation_history.append({"role": "user", "content": user_input})

        # Format the conversation using the template
        formatted_chat = tokenizer.apply_chat_template(
            conversation_history, tokenize=False, add_generation_prompt=True
        )

        # Tokenize and generate response
        inputs = tokenizer(
            formatted_chat, return_tensors="pt", add_special_tokens=False
        ).to(model.device)
        outputs = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, pad_token_id = tokenizer.eos_token_id)
        response = tokenizer.decode(
            outputs[0][len(inputs.input_ids[0]) :], skip_special_tokens=True
        )

        print("\nAssistant:", response)

        # Append assistant response to conversation history
        conversation_history.append({"role": "assistant", "content": response})


In [28]:
chat_session_with_rag(model, tokenizer, embedding_model, knowledge_base)

You are now chatting with the LLM. Type 'quit' to end the conversation.

You: What are some important dates in space travel history?

Assistant: * 1961: Yuri Gagarin becomes the first human in space.
* 1969: Apollo 11 lands humans on the Moon.
* 1977: The first space shuttle mission, STS-1, is launched.
* 1998: The Hubble Space Telescope is launched into orbit.
* 2003: The Mars Exploration Rovers Spirit and Opportunity land on Mars.
* 2006: The first private spaceflight is accomplished by SpaceX's Dragon capsule.

You: quit
